# 04 渠道归因：结构效应 vs 比率效应

## 本步目标

1. 把整体通过率变化拆成“渠道结构变化”和“同一渠道自身变化”。
2. 找出贡献最大的渠道。
3. 能用手算解释一个渠道的效应。

## 公式

```text
结构效应 =（当期占比 - 基期占比）×（基期渠道通过率 - 基期整体通过率）
比率效应 = 当期占比 ×（当期渠道通过率 - 基期渠道通过率）
总效应 = 结构效应 + 比率效应
```

注意：结构效应 + 比率效应 = 整体通过率变化，不是把“总效应”叫成“结构效应”。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display
from src.db import query_df
from src.charts import monthly_trend, channel_bar, contribution_bar, risk_scatter

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 50)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

## 1. 渠道贡献明细

In [2]:
decomp = query_df("SELECT * FROM `v_渠道归因` ORDER BY `总效应`")
display(decomp)

,渠道类型,基期占比,当期占比,基期通过率,当期通过率,结构效应,比率效应,总效应
0,Country-wide,0.1666,0.1217,0.9064,0.8764,-0.027462,-0.003652,-0.031114
1,Credit and cash offices,0.6026,0.6506,0.0856,0.0784,-0.010009,-0.004684,-0.014694
2,Stone,0.0547,0.0388,0.9219,0.8812,-0.010009,-0.001579,-0.011587
3,Regional / Local,0.0354,0.0299,0.9129,0.8744,-0.003429,-0.001149,-0.004578
4,AP+ (Cash loan),0.0085,0.0130,0.3909,0.2308,0.000434,-0.002081,-0.001647
5,Contact center,0.1246,0.1371,0.0309,0.0433,-0.003291,0.001701,-0.001591
6,Channel of corporate sales,0.0075,0.0089,0.2150,0.2288,-0.000108,0.000122,0.000015


## 2. 合计效应

In [3]:
totals = query_df('''
SELECT ROUND(SUM(`结构效应`),4) AS 结构效应合计,
       ROUND(SUM(`比率效应`),4) AS 比率效应合计,
       ROUND(SUM(`总效应`),4) AS 总效应合计
FROM `v_渠道归因`
''')
display(totals)
assert abs(float(totals.loc[0, "结构效应合计"]) + 0.0539) < 0.0005
assert abs(float(totals.loc[0, "比率效应合计"]) + 0.0113) < 0.0005
assert abs(float(totals.loc[0, "总效应合计"]) + 0.0652) < 0.0005
print("分解结果与预期一致")

,结构效应合计,比率效应合计,总效应合计
0,-0.0539,-0.0113,-0.0652


分解结果与预期一致


## 3. 贡献图

In [4]:
contribution_bar(decomp, FIG / "03_渠道贡献.png")

WindowsPath('E:/ai数据分析工作/consumer-finance-approval-analysis/outputs/figures/03_渠道贡献.png')

## 4. 手算示例：Credit and cash offices

In [5]:
cur = query_df("SELECT * FROM `v_渠道月度` WHERE `月份`='2026-09' AND `渠道类型`='Credit and cash offices'").iloc[0]
prev = query_df("SELECT * FROM `v_渠道月度` WHERE `月份`='2026-08' AND `渠道类型`='Credit and cash offices'").iloc[0]
cur_total = query_df("SELECT SUM(`进件量`) AS n FROM `v_渠道月度` WHERE `月份`='2026-09'").iloc[0,0]
prev_total = query_df("SELECT SUM(`进件量`) AS n FROM `v_渠道月度` WHERE `月份`='2026-08'").iloc[0,0]
prev_overall = query_df("SELECT `审批通过率` FROM `v_申请审批月度` WHERE `月份`='2026-08'").iloc[0,0]

weight_cur = cur["进件量"] / cur_total
weight_prev = prev["进件量"] / prev_total
mix = (weight_cur - weight_prev) * (prev["审批通过率"] - prev_overall)
rate = weight_cur * (cur["审批通过率"] - prev["审批通过率"])
print(f"当期占比={weight_cur:.4f}, 基期占比={weight_prev:.4f}")
print(f"结构效应={mix:.6f}, 比率效应={rate:.6f}, 合计={mix+rate:.6f}")

当期占比=0.6506, 基期占比=0.6026
结构效应=-0.010007, 比率效应=-0.004684, 合计=-0.014691


## 本步结论

通过率下降 6.52pp，其中结构效应 -5.39pp、比率效应 -1.13pp；低通过率渠道占比上升是主因，但各渠道自身也略有收紧。